In [8]:
import os
import shutil
import yaml
import pandas as pd
from sklearn.model_selection import KFold
from ultralytics import YOLO

# --- CONFIGURATION ---
# 1. Path to the dataset you created in the previous step
DATASET_ROOT = r"D:\projeto_placentas_clayton\dataset_ready_for_yolo"

# 2. Where to save the K-Fold experiments
PROJECT_DIR = r"D:\projeto_placentas_clayton\kfold_experiment"

# 3. Training Settings
# 'm' (Medium) is better for segmentation than 's' (Small), but uses more VRAM.
MODEL_WEIGHTS = 'yolov8s-seg.pt'  
BATCH_SIZE = 2        # STRICT LIMIT for 4GB VRAM with Medium model
IMG_SIZE = 1024       # High res for histology details
EPOCHS_PER_FOLD = 60  # 60 is enough per fold since we have little data
KSPLIT = 5            # 5-Fold Cross Validation
RANDOM_STATE = 42

# Create project dir if not exists
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Configuration loaded. Results will be saved to: {PROJECT_DIR}")

Configuration loaded. Results will be saved to: D:\projeto_placentas_clayton\kfold_experiment


In [9]:
# List to hold all image/label pairs
all_images = []
all_labels = []

print("Scanning dataset to consolidate files...")

# Scan both subfolders (train and valid) to get the full 66 images back together
for subdir in ['train', 'valid']:
    img_dir = os.path.join(DATASET_ROOT, subdir, 'images')
    lbl_dir = os.path.join(DATASET_ROOT, subdir, 'labels')
    
    if os.path.exists(img_dir):
        files = os.listdir(img_dir)
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif')):
                img_path = os.path.join(img_dir, f)
                # Construct label path (assuming same filename with .txt)
                label_name = os.path.splitext(f)[0] + ".txt"
                lbl_path = os.path.join(lbl_dir, label_name)
                
                # Only add if label exists
                if os.path.exists(lbl_path):
                    all_images.append(img_path)
                    all_labels.append(lbl_path)

print(f"Total valid pairs found: {len(all_images)}")

# Simple check to ensure lists are aligned
assert len(all_images) == len(all_labels), "Mismatch between images and labels!"

Scanning dataset to consolidate files...
Total valid pairs found: 66


In [10]:
# Initialize K-Fold
kf = KFold(n_splits=KSPLIT, shuffle=True, random_state=RANDOM_STATE)

results_summary = []

# Loop through each fold
for k, (train_idx, val_idx) in enumerate(kf.split(all_images)):
    fold_num = k + 1
    fold_name = f"fold_{fold_num}"
    
    # Define paths
    fold_run_dir = os.path.join(PROJECT_DIR, 'runs', fold_name)
    weights_path_best = os.path.join(fold_run_dir, 'weights', 'best.pt')
    weights_path_last = os.path.join(fold_run_dir, 'weights', 'last.pt') # Backup check
    
    fold_dataset_dir = os.path.join(PROJECT_DIR, fold_name, 'dataset')
    yaml_path = os.path.join(PROJECT_DIR, fold_name, 'data.yaml')

    print(f"\n{'='*40}")
    print(f"### STATUS: Checking {fold_name} ({fold_num}/{KSPLIT})")
    
    # --- 1. RESUME CHECK (Improved) ---
    # Check for best.pt OR last.pt
    model_to_load = None
    if os.path.exists(weights_path_best):
        model_to_load = weights_path_best
    elif os.path.exists(weights_path_last):
        print(f"   (Found 'last.pt' but not 'best.pt'. Using last.pt to skip training.)")
        model_to_load = weights_path_last

    if model_to_load:
        print(f"✅ Fold {fold_num} is ALREADY DONE. Weights found at: {model_to_load}")
        print(f"   ⏭️ Skipping training. Jumping to validation step...")
        
        try:
            model = YOLO(model_to_load)
            # Re-verify YAML exists for validation
            if not os.path.exists(yaml_path):
                 # Emergency rebuild of yaml if missing
                 yaml_content = {
                    'path': os.path.abspath(fold_dataset_dir).replace('\\', '/'),
                    'train': 'train/images',
                    'val': 'val/images',
                    'names': {0: 'microcotiledone'}
                }
                 with open(yaml_path, 'w') as f:
                    yaml.dump(yaml_content, f, default_flow_style=False)

            metrics = model.val(data=yaml_path, split='val', verbose=False, plots=False)
            results_summary.append({
                'Fold': fold_name,
                'Box_mAP50': metrics.box.map50,
                'Mask_mAP50': metrics.seg.map50,
                'Mask_mAP50-95': metrics.seg.map
            })
            continue # <--- THIS STOPS THE LOOP HERE AND MOVES TO FOLD 2
        except Exception as e:
            print(f"   ⚠️ Warning: Could not load existing weights. Retraining. Error: {e}")

    # --- 2. DATASET SETUP ---
    print(f"   -> 🛠️ Setting up new dataset for {fold_name}...")
    if os.path.exists(fold_dataset_dir):
        shutil.rmtree(fold_dataset_dir) 
        
    for split in ['train', 'val']:
        os.makedirs(os.path.join(fold_dataset_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(fold_dataset_dir, split, 'labels'), exist_ok=True)

    for i in train_idx:
        shutil.copy(all_images[i], os.path.join(fold_dataset_dir, 'train', 'images'))
        shutil.copy(all_labels[i], os.path.join(fold_dataset_dir, 'train', 'labels'))
    for i in val_idx:
        shutil.copy(all_images[i], os.path.join(fold_dataset_dir, 'val', 'images'))
        shutil.copy(all_labels[i], os.path.join(fold_dataset_dir, 'val', 'labels'))

    yaml_content = {
        'path': os.path.abspath(fold_dataset_dir).replace('\\', '/'),
        'train': 'train/images',
        'val': 'val/images',
        'names': {0: 'microcotiledone'}
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)

    # --- 3. TRAINING ---
    print(f"   -> 🚀 STARTING TRAINING FOR {fold_name}")
    model = YOLO(MODEL_WEIGHTS) 
    
    model.train(
        data=yaml_path,
        epochs=EPOCHS_PER_FOLD,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=10,
        device=0,
        project=os.path.join(PROJECT_DIR, 'runs'),
        name=fold_name,
        exist_ok=True,
        verbose=True,
        amp=True,
        workers=0,
        plots=False,
        val=True  # ENABLED so best.pt is created this time
    )
    
    # --- 4. VALIDATION ---
    print(f"   -> Validating {fold_name}...")
    metrics = model.val(split='val', plots=False)
    
    results_summary.append({
        'Fold': fold_name,
        'Box_mAP50': metrics.box.map50,
        'Mask_mAP50': metrics.seg.map50,
        'Mask_mAP50-95': metrics.seg.map
    })
    
    print(f">>> {fold_name} FINISHED. Moving to next fold.")

print("\nK-Fold Cross-Validation Pipeline Finished!")


### STATUS: Checking fold_1 (1/5)
✅ Fold 1 is ALREADY DONE. Weights found at: D:\projeto_placentas_clayton\kfold_experiment\runs\fold_1\weights\best.pt
   ⏭️ Skipping training. Jumping to validation step...
Ultralytics 8.3.232  Python-3.10.19 torch-2.3.1 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients
val: Fast image access  (ping: 0.10.0 ms, read: 341.078.7 MB/s, size: 51.6 KB)
val: Scanning D:\projeto_placentas_clayton\kfold_experiment\fold_1\dataset\val\labels.cache... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 14.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 17.4s/it 17.4s
                   all         14        493       0.87      0.884      0.902      0.617      0.879       0.87      0.896      0.529
Speed: 14.6ms preprocess, 698.1ms inference, 0.0ms loss, 13.3ms pos

In [ ]:
PROJECT_DIR = r"D:\projeto_placentas_clayton\kfold_experiment"

print("Checking for existing weights...")
for i in range(1, 6):
    fold_name = f"fold_{i}"
    # The path we are checking in the script
    expected_path = os.path.join(PROJECT_DIR, 'runs', fold_name, 'weights', 'best.pt')
    
    if os.path.exists(expected_path):
        print(f"✅ {fold_name}: Found best.pt")
    else:
        print(f"❌ {fold_name}: No best.pt found (Needs training)")
        
        # Check if maybe the folder exists but is empty/corrupt
        fold_dir = os.path.join(PROJECT_DIR, 'runs', fold_name)
        if os.path.exists(fold_dir):
            print(f"   (Folder '{fold_name}' exists, but training didn't finish.)")